# Delphi-lite demo: which movies will a user rate on their next active day?

Everything below uses **held-out test users**: their histories were never used for training, for choosing the model, or for tuning the baselines.

For one user we reveal their ratings up to the end of an active day and rank the 10,000 movies for the user's **next active day**. The model sees only days before that day (its most recent 256 ratings); movies the user already rated are never suggested. Next to Delphi we show **RECENT-POP**, the strongest baseline: what training users rated most in the last 30 days.

Rows marked `hit = True` are movies the user really rated on that next day. `p` is the model's probability for the movie among all candidates.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')  # run from the repository root
sys.path.insert(0, os.getcwd())
import pandas as pd
from IPython.display import HTML, display
from moviedelphi.demo import Explorer

ex = Explorer('models/delphi_final.pt', split='test', device='cpu')
print(f'{len(ex.users)} held-out test users with at least two active days')

7129 held-out test users with at least two active days


In [2]:
def highlight(df):
    if 'hit' not in df:
        return df.style
    return df.style.apply(lambda r: ['background-color:#dcefdc' if r.hit else '' for _ in r], axis=1)

def side_by_side(*frames):
    html = ''.join(f'<div style="margin-right:24px"><b>{t}</b>{highlight(f).to_html()}</div>' for t, f in frames)
    display(HTML(f'<div style="display:flex;align-items:flex-start">{html}</div>'))

def show(r, n_history=10):
    print(f"history: {len(r['history'])} ratings up to {r['last_day']}  ->  next active day {r['target_day']}"
          f" ({len(r['target'])} movies)")
    print(f"recall@20   Delphi {r['model_recall@20']:.2f}   RECENT-POP {r['baseline_recall@20']:.2f}")
    side_by_side((f'Last {n_history} ratings seen', r['history'].tail(n_history)),
                 ('What the user rated next', r['target'].head(20)))
    side_by_side(('Delphi top 20', r['model']), ('RECENT-POP top 20', r['baseline']))

## 1. Step through one user's timeline

Each row reveals one more active day and predicts the next one. This is the comparison tool: how predictions change with more visible history, against the baseline.

In [3]:
USER = 759  # index into ex.users; try any number below len(ex.users)
ex.timeline(USER)

,revealed days,events seen,next day,movies that day,Delphi recall@20,RECENT-POP recall@20
0,1,39,2006-10-27,32,0.300000,0.30
1,2,71,2006-10-29,1,1.000000,1.00
2,3,72,2006-11-02,1,0.000000,0.00
3,4,73,2006-11-29,1,0.000000,1.00
4,5,74,2006-12-05,1,0.000000,0.00
5,6,75,2007-02-10,27,0.100000,0.35
6,7,102,2007-03-17,3,0.666667,0.00
7,8,105,2007-06-01,1,1.000000,1.00
8,9,106,2007-06-07,2,0.000000,0.00


## 2. Inspect one prediction

This user's history is mainstream blockbusters (*Jaws*, *Star Wars*, *Men in Black*, *Jurassic Park*). Five weeks later they rated both *Kill Bill* films. Delphi ranks both in its top 20, while RECENT-POP, which only knows what everyone rated that month, offers *Little Miss Sunshine*, *Children of Men* and *Casino Royale*. Both *Kill Bill* films came out years before the prediction day, so this success does not come from knowing later releases.

In [4]:
show(ex.predict_next_day(USER, n_days=7))

history: 102 ratings up to 2007-02-10  ->  next active day 2007-03-17 (3 movies)
recall@20   Delphi 0.67   RECENT-POP 0.00


,date,title,genres
92,2007-02-10,Star Wars: Episode I - The Phantom Menace (1999),Action|Adventure|Sci-Fi
93,2007-02-10,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Romance
94,2007-02-10,Demolition Man (1993),Action|Adventure|Sci-Fi
95,2007-02-10,Sleepless in Seattle (1993),Comedy|Drama|Romance
96,2007-02-10,Home Alone (1990),Children|Comedy
97,2007-02-10,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
98,2007-02-10,Men in Black (a.k.a. MIB) (1997),Action|Comedy|Sci-Fi
99,2007-02-10,Pretty Woman (1990),Comedy|Romance
100,2007-02-10,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi
101,2007-02-10,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller


,title,genres,p,hit
0,Batman Begins (2005),Action|Crime|IMAX,0.005900,False
1,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.005200,False
2,Memento (2000),Mystery|Thriller,0.004800,False
3,"Matrix, The (1999)",Action|Sci-Fi|Thriller,0.004700,False
4,Casino Royale (2006),Action|Adventure|Thriller,0.004600,False
5,Sin City (2005),Action|Crime|Film-Noir|Mystery|Thriller,0.004500,False
6,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller,0.004400,True
7,Little Miss Sunshine (2006),Adventure|Comedy|Drama,0.004400,False
8,Ocean's Eleven (2001),Crime|Thriller,0.004300,False
9,"Lord of the Rings: The Fellowship of the Ring, The (2001)",Adventure|Fantasy,0.004200,False


## 3. A failure, explained

One day of history: 17 ratings, mostly of 1995 releases (*GoldenEye*, *Heat*, *Twelve Monkeys*). A week later the user rated films that everyone was rating then (*Pulp Fiction*, *Apollo 13*, *Batman*, *True Lies*), and RECENT-POP has five of the six in its top 20. Delphi followed the user's apparent taste for action and thrillers (*The Rock*, *Mission: Impossible*, *Speed*) and found none. On average Delphi does better than RECENT-POP for histories of this length, so this is a miss of personalisation, not the typical case. Misses are common: in 42% of test cases neither list contains a single movie the user rated next.

In [5]:
show(ex.predict_next_day(5393, n_days=1))

history: 17 ratings up to 1996-10-11  ->  next active day 1996-10-18 (6 movies)
recall@20   Delphi 0.00   RECENT-POP 0.83


,date,title,genres
7,1996-10-11,Nixon (1995),Drama
8,1996-10-11,Sense and Sensibility (1995),Drama|Romance
9,1996-10-11,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
10,1996-10-11,"Bridges of Madison County, The (1995)",Drama|Romance
11,1996-10-11,Heat (1995),Action|Crime|Thriller
12,1996-10-11,Jumanji (1995),Adventure|Children|Fantasy
13,1996-10-11,Get Shorty (1995),Comedy|Crime|Thriller
14,1996-10-11,"American President, The (1995)",Comedy|Drama|Romance
15,1996-10-11,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller
16,1996-10-11,Casino (1995),Crime|Drama


,title,genres,p,hit
0,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.005700,False
1,"Rock, The (1996)",Action|Adventure|Thriller,0.005100,False
2,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,0.005000,False
3,Mission: Impossible (1996),Action|Adventure|Mystery|Thriller,0.004900,False
4,Home Alone (1990),Children|Comedy,0.004800,False
5,Forrest Gump (1994),Comedy|Drama|Romance|War,0.004800,False
6,Speed (1994),Action|Romance|Thriller,0.004700,False
7,Terminator 2: Judgment Day (1991),Action|Sci-Fi,0.004700,False
8,Ghost (1990),Comedy|Drama|Fantasy|Romance|Thriller,0.004600,False
9,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,0.004500,False


## 4. Your own history (new data)

Find titles with `ex.find(...)`, then give a list of exact titles and the date you rated them (the data covers 1995-01-09 to 2019-11-21; later dates are moved to the last day).

In [6]:
ex.find('matrix').head(5)

,display_title,genres,train_count
token,,,
6,"Matrix, The (1999)",Action|Sci-Fi|Thriller,58226
157,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,18408
244,"Matrix Revolutions, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,14449
1470,"Animatrix, The (2003)",Action|Animation|Drama|Sci-Fi,3260


In [7]:
my_movies = ['Matrix, The (1999)', 'Inception (2010)', 'Interstellar (2014)', 'Arrival (2016)']
delphi_top, recent_pop_top = ex.predict_for_titles(my_movies, '2019-06-01', k=10)
side_by_side(('Delphi top 10', delphi_top), ('RECENT-POP top 10', recent_pop_top))

,title,genres,p
0,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX,0.003600
1,WALL·E (2008),Adventure|Animation|Children|Romance|Sci-Fi,0.003300
2,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,0.003100
3,Django Unchained (2012),Action|Drama|Western,0.003100
4,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.003100
5,Pirates of the Caribbean: The Curse of the Black Pearl (2003),Action|Adventure|Comedy|Fantasy,0.003000
6,The Martian (2015),Adventure|Drama|Sci-Fi,0.003000
7,Avatar (2009),Action|Adventure|Sci-Fi|IMAX,0.003000
8,"Wolf of Wall Street, The (2013)",Comedy|Crime|Drama,0.003000
9,Up (2009),Adventure|Animation|Children|Drama,0.003000


### How to read this

* A hit is only a movie the user *rated* that day; MovieLens dates are rating dates, not watch dates, so a sensible suggestion can still count as a miss.
* The model never sees the next day's date, only the dates of earlier ratings.
* Users were split, not time: Delphi learned from training users' later ratings, so it sometimes suggests a film that appeared after the prediction day (5% of its top 20, 2% of its hits). Without such films its gain over RECENT-POP stays +0.009.
* Test-set averages, not single cases, are the evidence: see the report.